# Setup e Imports

In [2]:
import os
import sys
import torch.nn as nn
import torch.optim as optim
import pandas as pd

# Descobre onde o notebook está rodando
current_dir = os.getcwd()

# Se estiver rodando dentro da pasta 'experiments', adiciona a pasta raiz ao path
if current_dir.endswith('experiments'):
    sys.path.append(os.path.abspath(os.path.join(current_dir, '..')))
else:
    sys.path.append(current_dir)

from core.config import DEVICE, DATASET_FINAL
from core.data import get_target_splits, get_dataloaders
from core.models import get_vibnet_model
from core.engine import pre_train_source, train_target_fold

print(f"Usando dispositivo: {DEVICE}")

Usando dispositivo: cuda


# Pré-treinamento (Source)

In [ ]:
# Roda apenas uma vez (ou pula se os pesos já existirem)
w_imagenet = pre_train_source(target_to_exclude="PU", start_with_imagenet=True)
w_scratch = pre_train_source(target_to_exclude="PU", start_with_imagenet=False)

# Configuração do Target

In [ ]:
# Lista as condições
pu_root = os.path.join(DATASET_FINAL, "PU")
conditions = sorted([d for d in os.listdir(pu_root) if os.path.isdir(os.path.join(pu_root, d))])
strategies = ["Scratch", "ImageNet", "VibNet_from_Scratch", "VibNet_from_ImageNet"]

print(f"Condições encontradas: {conditions}")

# Experimento

In [ ]:
results = []

for test_cond in conditions:
    print(f"\n{'='*40}\nFold: Testando em {test_cond}\n{'='*40}")
    
    train_x, train_y, test_x, test_y, num_classes = get_target_splits("PU", test_cond)
    dataloaders = get_dataloaders(train_x, train_y, test_x, test_y)

    for strat in strategies:
        print(f"-> Estratégia: {strat}")
        
        w_path = w_imagenet if "ImageNet" in strat else w_scratch
        model = get_vibnet_model(num_classes, strat, w_path)
        
        lr = 0.001 if strat == "Scratch" else 0.0001
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        
        metrics = train_target_fold(model, dataloaders, optimizer, criterion, epochs=8)
        print(f"   Bal Acc: {metrics['Bal Accuracy']:.4f} | F1: {metrics['Macro F1']:.4f}")
        
        results.append({"Condition": test_cond, "Strategy": strat, **metrics})

# Resultados

In [ ]:
df = pd.DataFrame(results)

resumo = df.groupby("Strategy")[["Bal Accuracy", "Macro F1"]].agg(['mean', 'std'])
display(resumo)

# Salva para anexar aos manuscritos depois
df.to_csv("resultados_pu_unificado.csv", index=False)